# Building different tokenizer models from scratch


In [38]:
from tokenizers import Tokenizer, AddedToken, pre_tokenizers
from tokenizers.models import BPE, WordPiece, Unigram
from tokenizers.trainers import BpeTrainer, WordPieceTrainer, UnigramTrainer
from tokenizers.pre_tokenizers import PreTokenizer, Whitespace
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence

In [22]:
# träningsfiler
files = ["svensk_text_1.txt",
        "svensk_text_2.txt",
        "svensk_text_3.txt",
        ]

# variables
vocab_size = 10000 
min_frequency = 4

In [28]:
text = "Det här är en exempelmening på svenska med åäö och sammansatta ord som e-post."
text = "Elmontörer kommer att spela en viktig roll i framtidens samhälle."
text = "Elmaterial har en viktig roll i framtidens samhälle att spela."
# text = "Byggmontör kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggarbete kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggmaterial kommer att spela en viktig roll i framtidens samhälle."
text = "Han behöver elmateral för sitt elarbete som elmontör."
text

'Han behöver elmateral för sitt elarbete som elmontör.'

### WordPiece model like BERT

In [29]:
# Build a tokenizer
bert_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
bert_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
bert_tokenizer.pre_tokenizer = Whitespace()


# Specialtoken är oftast desamma
special_tokens = [
    "[PAD]", # Padding-token
    "[UNK]", # Okänd token
    "[CLS]", # Klassificeringstoken
    "[SEP]", # Separator-token
    "[MASK]", # Maskeringstoken
]

# Initilize trainer
trainer = WordPieceTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=special_tokens,
    continuing_subword_prefix="##",
)

bert_tokenizer.train(files, trainer)

# --- 7. Spara tokenizern ---
# Sparar konfigurationen och den tränade vokabulären till en JSON-fil.
bert_output_path = "bert-custom-tokenizer.json"
bert_tokenizer.save(bert_output_path)

print(f"Tokenizer tränad och sparad till {bert_output_path}")
print(f"Vocab size after training: {bert_tokenizer.get_vocab_size()}")



Tokenizer tränad och sparad till bert-custom-tokenizer.json
Vocab size after training: 2516



### BPE model like GPT-2

In [35]:
# Build a tokenizer
bpe_tokenizer = Tokenizer(BPE())
bpe_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
bpe_tokenizer.pre_tokenizer = Whitespace()

# Specialtoken är oftast desamma
special_tokens=["<|endoftext|>"]

# Initilize trainer
trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=special_tokens,
)

bpe_tokenizer.train(files, trainer)

# --- 7. Spara tokenizern ---
# Sparar konfigurationen och den tränade vokabulären till en JSON-fil.
bpe_output_path = "bpe-custom-tokenizer.json"
bpe_tokenizer.save(bpe_output_path)

print(f"Tokenizer tränad och sparad till {bpe_output_path}")
print(f"Vocab size after training: {bpe_tokenizer.get_vocab_size()}")



Tokenizer tränad och sparad till bpe-custom-tokenizer.json
Vocab size after training: 2221



### Unigram model like Albert

In [36]:
uni_tokenizer = Tokenizer(Unigram())
uni_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
uni_tokenizer.pre_tokenizer = Whitespace()

# Specialtoken är desamma
special_tokens = [
    "[PAD]",
    "[UNK]",
    "[CLS]",
    "[SEP]",
    "[MASK]",
]

# Initiera UnigramTrainer
trainer = UnigramTrainer(
    vocab_size=vocab_size,
    special_tokens=special_tokens,
    # Unigram behöver veta vilken token som är UNK
    unk_token="[UNK]",
)


uni_tokenizer.train(files, trainer)

# --- 7. Spara tokenizern ---
# Sparar konfigurationen och den tränade vokabulären till en JSON-fil.
uni_output_path = "uni-custom-tokenizer.json"
uni_tokenizer.save(uni_output_path)

print(f"Tokenizer tränad och sparad till {uni_output_path}")
print(f"Vocab size after training: {uni_tokenizer.get_vocab_size()}")



Tokenizer tränad och sparad till uni-custom-tokenizer.json
Vocab size after training: 2422


## Load tokenizer

In [47]:
# Restoring model from learned config/vocab
loaded_bert_tokenizer = Tokenizer.from_file(bert_output_path)
loaded_bpe_tokenizer = Tokenizer.from_file(bpe_output_path)
loaded_uni_tokenizer = Tokenizer.from_file(uni_output_path)

# Test encoding
bert_encode = loaded_uni_tokenizer.encode(text)
bpe_encode = loaded_uni_tokenizer.encode(text)
uni_encode = loaded_uni_tokenizer.encode(text)

# token information
print("Originaltext:", text)
print("BERT Tokens:", bert_encode.tokens)
print("BPE Tokens:", bpe_encode.tokens)
print("Unicode Tokens:", uni_encode.tokens)

Originaltext: Han behöver elmateral för sitt elarbete som elmontör.
BERT Tokens: ['ha', 'n', 'behöv', 'er', 'el', 'mate', 'r', 'al', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']
BPE Tokens: ['ha', 'n', 'behöv', 'er', 'el', 'mate', 'r', 'al', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']
Unicode Tokens: ['ha', 'n', 'behöv', 'er', 'el', 'mate', 'r', 'al', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']
